In [3]:
import pandas as pd

ratings = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user","movie","rating","time"]
)


In [4]:
pip install torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import random
import torch
from torch.utils.data import Dataset

class MovieDataset(Dataset):
    def __init__(self, df, n_movies, num_neg=2):
        self.users = df.user_idx.values
        self.movies = df.movie_idx.values
        self.n_movies = n_movies
        self.num_neg = num_neg

        # Build user → watched set
        self.user_pos = {}
        for u, m in zip(self.users, self.movies):
            self.user_pos.setdefault(u, set()).add(m)

        self.data = []
        self._prepare()

    def _prepare(self):
        for u, m in zip(self.users, self.movies):
            # positive sample
            self.data.append((u, m, 1))

            # negative samples
            for _ in range(self.num_neg):
                neg_m = random.randint(0, self.n_movies - 1)
                while neg_m in self.user_pos[u]:
                    neg_m = random.randint(0, self.n_movies - 1)
                self.data.append((u, neg_m, 0))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        u, m, l = self.data[idx]
        return (
            torch.tensor(u),
            torch.tensor(m),
            torch.tensor(l, dtype=torch.float32)
        )

In [6]:
import pandas as pd

# Load ratings again
ratings = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user","movie","rating","time"]
)

# Convert to implicit feedback
ratings["label"] = (ratings["rating"] >= 4).astype(int)
ratings = ratings[ratings["label"] == 1]

# Create user and movie index mapping
user_map = {u:i for i,u in enumerate(ratings.user.unique())}
movie_map = {m:i for i,m in enumerate(ratings.movie.unique())}

ratings["user_idx"] = ratings["user"].map(user_map)
ratings["movie_idx"] = ratings["movie"].map(movie_map)

n_users = len(user_map)
n_movies = len(movie_map)

# Train test split
from sklearn.model_selection import train_test_split
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

print("Train size:", len(train_df))

Train size: 460224


In [9]:
from torch.utils.data import DataLoader
train_dataset = MovieDataset(train_df, n_movies)
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)

print("Training samples:", len(train_dataset))

Training samples: 1380672


In [10]:
import torch.nn as nn

class Recommender(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=64):
        super().__init__()

        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)

        self.fc = nn.Sequential(
            nn.Linear(emb_dim*2, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, user, movie):
        u = self.user_emb(user)
        m = self.movie_emb(movie)
        x = torch.cat([u, m], dim=1)
        return self.fc(x)

In [11]:
model = Recommender(n_users, n_movies)

loss_fn = torch.nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [12]:
for epoch in range(5):
    total_loss = 0

    for users, movies, labels in train_loader:
        preds = model(users, movies).squeeze()
        loss = loss_fn(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        

        total_loss += loss.item()

    print("Epoch", epoch, "Loss:", total_loss)

Epoch 0 Loss: 634.0507885217667


KeyboardInterrupt: 

In [17]:
model.eval()

# pick a user
user_id = 0

# movies this user already watched
watched = set(train_df[train_df.user_idx == user_id].movie_idx.values)

user_tensor = torch.tensor([user_id] * n_movies)
movie_tensor = torch.arange(n_movies)

with torch.no_grad():
    scores = model(user_tensor, movie_tensor).squeeze()

# remove watched movies
scores_filtered = []
for i, s in enumerate(scores):
    if i not in watched:
        scores_filtered.append((i, s.item()))

# sort
scores_filtered.sort(key=lambda x: x[1], reverse=True)

top_movies = scores_filtered[:10]

print("Top movie indices:", [x[0] for x in top_movies])

Top movie indices: [84, 30, 52, 87, 73, 40, 88, 161, 147, 109]


In [18]:
idx_to_movie = {v:k for k,v in movie_map.items()}

recommended_ids = [idx_to_movie[i] for i,_ in top_movies]

movies = pd.read_csv(
    "../data/ml-1m/movies.dat",
    sep="::",
    engine="python",
    names=["movie","title","genre"],
    encoding="latin-1"
)

print(movies[movies["movie"].isin(recommended_ids)][["movie","title"]])

      movie                                              title
315     318                   Shawshank Redemption, The (1994)
589     593                   Silence of the Lambs, The (1991)
1178   1196  Star Wars: Episode V - The Empire Strikes Back...
1192   1210  Star Wars: Episode VI - Return of the Jedi (1983)
1220   1240                             Terminator, The (1984)
1959   2028                         Saving Private Ryan (1998)
2502   2571                                 Matrix, The (1999)
2693   2762                            Sixth Sense, The (1999)
2789   2858                             American Beauty (1999)
2928   2997                        Being John Malkovich (1999)


In [ ]:
-----------------------------------------------------------------------------------------------------------------------------

In [19]:
def hit_rate_at_k(model, train_df, test_df, n_movies, k=10):
    model.eval()

    hits = 0
    total = 0

    # group test data by user
    user_test_groups = test_df.groupby("user_idx")

    for user_id, group in user_test_groups:

        # take one true movie from test set
        true_movie = group.movie_idx.values[0]

        # movies already watched in training
        watched = set(train_df[train_df.user_idx == user_id].movie_idx.values)

        # score all movies
        user_tensor = torch.tensor([user_id] * n_movies)
        movie_tensor = torch.arange(n_movies)

        with torch.no_grad():
            scores = model(user_tensor, movie_tensor).squeeze()

        # filter watched movies
        candidates = []
        for i, s in enumerate(scores):
            if i not in watched:
                candidates.append((i, s.item()))

        # sort by score
        candidates.sort(key=lambda x: x[1], reverse=True)

        # top-k movies
        top_k = [x[0] for x in candidates[:k]]

        if true_movie in top_k:
            hits += 1

        total += 1

    return hits / total

In [21]:
hr = hit_rate_at_k(model, train_df, test_df, n_movies, k=10)
print("Hit Rate@10:", hr)

Hit Rate@10: 0.07673060884070058


In [22]:
def precision_at_k(model, train_df, test_df, n_movies, k=10):
    model.eval()

    precisions = []

    user_test_groups = test_df.groupby("user_idx")

    for user_id, group in user_test_groups:

        true_movies = set(group.movie_idx.values)

        watched = set(train_df[train_df.user_idx == user_id].movie_idx.values)

        user_tensor = torch.tensor([user_id] * n_movies)
        movie_tensor = torch.arange(n_movies)

        with torch.no_grad():
            scores = model(user_tensor, movie_tensor).squeeze()

        candidates = []
        for i, s in enumerate(scores):
            if i not in watched:
                candidates.append((i, s.item()))

        candidates.sort(key=lambda x: x[1], reverse=True)

        top_k = [x[0] for x in candidates[:k]]

        hits = len(set(top_k) & true_movies)
        precisions.append(hits / k)

    return sum(precisions) / len(precisions)

In [ ]:
p = precision_at_k(model, train_df, test_df, n_movies, k=10)
print("Precision@10:", p)

In [ ]:
def recall_at_k(model, train_df, test_df, n_movies, k=10):
    model.eval()

    recalls = []

    user_test_groups = test_df.groupby("user_idx")

    for user_id, group in user_test_groups:

        true_movies = set(group.movie_idx.values)

        watched = set(train_df[train_df.user_idx == user_id].movie_idx.values)

        user_tensor = torch.tensor([user_id] * n_movies)
        movie_tensor = torch.arange(n_movies)

        with torch.no_grad():
            scores = model(user_tensor, movie_tensor).squeeze()

        candidates = []
        for i, s in enumerate(scores):
            if i not in watched:
                candidates.append((i, s.item()))

        candidates.sort(key=lambda x: x[1], reverse=True)

        top_k = [x[0] for x in candidates[:k]]

        hits = len(set(top_k) & true_movies)

        if len(true_movies) > 0:
            recalls.append(hits / len(true_movies))

    return sum(recalls) / len(recalls)

In [ ]:
r = recall_at_k(model, train_df, test_df, n_movies, k=10)
print("Recall@10:", r)

In [2]:
# Reload full ratings with timestamp
ratings_full = pd.read_csv(
    "../data/ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["user","movie","rating","time"]
)

# Keep only positive interactions
ratings_full["label"] = (ratings_full["rating"] >= 4).astype(int)
ratings_full = ratings_full[ratings_full["label"] == 1]

# Create index mapping again
user_map = {u:i for i,u in enumerate(ratings_full.user.unique())}
movie_map = {m:i for i,m in enumerate(ratings_full.movie.unique())}

ratings_full["user_idx"] = ratings_full["user"].map(user_map)
ratings_full["movie_idx"] = ratings_full["movie"].map(movie_map)

n_users = len(user_map)
n_movies = len(movie_map)

print("Users:", n_users)
print("Movies:", n_movies)

NameError: name 'pd' is not defined

In [24]:
# Sort by user and time
ratings_full = ratings_full.sort_values(["user_idx", "time"])

ratings_full.head()

,user,movie,rating,time,label,user_idx,movie_idx
31,1,3186,4,978300019,1,0,23
22,1,1270,5,978300055,1,0,16
27,1,1721,4,978300055,1,0,20
37,1,1022,5,978300055,1,0,29
36,1,1836,5,978300172,1,0,28


In [25]:
sequence_length = 3

train_sequences = []
test_sequences = []

ratings_full = ratings_full.sort_values(["user_idx", "time"])

for user_id, group in ratings_full.groupby("user_idx"):
    
    movies = group["movie_idx"].tolist()
    
    if len(movies) <= sequence_length:
        continue
    
    # training sequences
    for i in range(len(movies) - sequence_length - 1):
        input_seq = movies[i:i+sequence_length]
        target = movies[i+sequence_length]
        train_sequences.append((user_id, input_seq, target))
    
    # test sequence
    input_seq = movies[-sequence_length-1:-1]
    target = movies[-1]
    test_sequences.append((user_id, input_seq, target))

print("Train:", len(train_sequences))
print("Test:", len(test_sequences))

Train: 551136
Test: 6035


In [26]:
class SequenceDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        user_id, seq, target = self.sequences[idx]
        return (
            torch.tensor(user_id, dtype=torch.long),
            torch.tensor(seq, dtype=torch.long),
            torch.tensor(target, dtype=torch.long)
        )

In [27]:
train_dataset = SequenceDataset(train_sequences)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

In [14]:
import torch.nn as nn

class LSTMRecommender(nn.Module):
    def __init__(self, n_movies, emb_dim=64, hidden_dim=128):
        super().__init__()
        
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, n_movies)

    def forward(self, x):
        x = self.movie_emb(x)           # (batch, seq_len, emb_dim)
        _, (hidden, _) = self.lstm(x)   # hidden: (1, batch, hidden_dim)
        output = self.fc(hidden.squeeze(0))  # (batch, n_movies)
        return output

In [15]:
seq_model = LSTMRecommender(n_movies)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(seq_model.parameters(), lr=0.001)

In [ ]:
for epoch in range(3):
    total_loss = 0
    
    for user_id, seq, target in train_loader:
        
        preds = seq_model(seq)
        loss = criterion(preds, target)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print("Epoch", epoch, "Loss:", total_loss)

In [ ]:
def sequence_hit_rate_test(model, test_sequences, k=10):
    model.eval()
    
    hits = 0
    total = 0
    
    with torch.no_grad():
        for seq, target in test_sequences:
            
            seq_tensor = torch.tensor(seq).unsqueeze(0)
            preds = model(seq_tensor)
            
            top_k = torch.topk(preds.squeeze(), k).indices.tolist()
            
            if target in top_k:
                hits += 1
            
            total += 1
    
    return hits / total

seq_hr_test = sequence_hit_rate_test(seq_model, test_sequences, k=10)
print("Proper Sequence Model Hit Rate@10:", seq_hr_test)

In [ ]:
sample_sequences = sequences[:5000]

seq_hr = sequence_hit_rate_at_k(seq_model, sample_sequences, k=10)
print("Sequence Model Hit Rate@10:", seq_hr)

In [ ]:
torch.save(model.state_dict(), "ncf_model.pth")
torch.save(seq_model.state_dict(), "lstm_model.pth")

In [ ]:
# Build dictionary: user → last sequence
user_last_sequence = {}

ratings_full = ratings_full.sort_values(["user_idx", "time"])

for user_id, group in ratings_full.groupby("user_idx"):
    movies = group["movie_idx"].tolist()
    
    if len(movies) > sequence_length:
        user_last_sequence[user_id] = movies[-sequence_length:]
        
print("Users with sequences:", len(user_last_sequence))

In [ ]:
def hybrid_hit_rate(ncf_model, lstm_model, train_df, test_sequences, n_movies, alpha=0.5, k=10):
    ncf_model.eval()
    lstm_model.eval()
    
    hits = 0
    total = 0
    
    with torch.no_grad():
        for seq, target in test_sequences:
            
            # get user_id from sequence (we assume test_sequences built in same order as grouping)
            # Instead we reconstruct using last element before target
            
            # find user whose last sequence matches
            # simpler: iterate per user instead
            
            seq_tensor = torch.tensor(seq).unsqueeze(0)
            
            # LSTM scores
            lstm_scores = lstm_model(seq_tensor).squeeze()
            
            # Need user_id — we recover from sequence via search
            # Instead, better pass user_id explicitly (we'll fix next)
            
            total += 1
    
    return 0

In [ ]:
------hybrid------

In [ ]:
def hybrid_hit_rate(ncf_model, lstm_model, train_df, test_sequences, n_movies, alpha=0.5, k=10):
    ncf_model.eval()
    lstm_model.eval()
    
    hits = 0
    total = 0
    
    with torch.no_grad():
        for user_id, seq, target in test_sequences:
            
            # NCF scores
            user_tensor = torch.tensor([user_id] * n_movies)
            movie_tensor = torch.arange(n_movies)
            ncf_scores = ncf_model(user_tensor, movie_tensor).squeeze()
            
            # LSTM scores
            seq_tensor = torch.tensor(seq).unsqueeze(0)
            lstm_scores = lstm_model(seq_tensor).squeeze()
            
            # Normalize both
            ncf_scores = (ncf_scores - ncf_scores.min()) / (ncf_scores.max() - ncf_scores.min() + 1e-8)
            lstm_scores = (lstm_scores - lstm_scores.min()) / (lstm_scores.max() - lstm_scores.min() + 1e-8)
            
            # Combine
            combined_scores = alpha * ncf_scores + (1 - alpha) * lstm_scores
            
            # Remove watched movies
            watched = set(train_df[train_df.user_idx == user_id].movie_idx.values)
            
            candidates = []
            for i, s in enumerate(combined_scores):
                if i not in watched:
                    candidates.append((i, s.item()))
            
            candidates.sort(key=lambda x: x[1], reverse=True)
            top_k = [x[0] for x in candidates[:k]]
            
            if target in top_k:
                hits += 1
            
            total += 1
    
    return hits / total

In [ ]:
hybrid_hr = hybrid_hit_rate(model, seq_model, train_df, test_sequences, n_movies, alpha=0.5, k=10)
print("Hybrid Hit@10:", hybrid_hr)

In [ ]:
Hybrid hit rate not enough, now gonna make ncf model better to match with lstm model

In [ ]:
# Chronological split for NCF

ratings_full = ratings_full.sort_values(["user_idx", "time"])

train_ncf = []
test_ncf = []

for user_id, group in ratings_full.groupby("user_idx"):
    
    movies = group["movie_idx"].tolist()
    
    if len(movies) < 2:
        continue
    
    # last movie for testing
    test_movie = movies[-1]
    
    # rest for training
    for m in movies[:-1]:
        train_ncf.append((user_id, m))
    
    test_ncf.append((user_id, test_movie))

print("NCF Train samples:", len(train_ncf))
print("NCF Test samples:", len(test_ncf))

In [ ]:
class NCFDataset(Dataset):
    def __init__(self, interactions, n_movies, num_neg=4):
        self.data = []
        self.n_movies = n_movies
        self.num_neg = num_neg
        
        user_pos = {}
        for u, m in interactions:
            user_pos.setdefault(u, set()).add(m)
        
        for u, m in interactions:
            self.data.append((u, m, 1))
            
            for _ in range(self.num_neg):
                neg = torch.randint(0, n_movies, (1,)).item()
                while neg in user_pos[u]:
                    neg = torch.randint(0, n_movies, (1,)).item()
                self.data.append((u, neg, 0))
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        u, m, l = self.data[idx]
        return (
            torch.tensor(u),
            torch.tensor(m),
            torch.tensor(l, dtype=torch.float32)
        )

In [ ]:
train_dataset_ncf = NCFDataset(train_ncf, n_movies, num_neg=4)
train_loader_ncf = DataLoader(train_dataset_ncf, batch_size=1024, shuffle=True)

print("Total NCF training samples:", len(train_dataset_ncf))

In [ ]:
class StrongNCF(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=128):
        super().__init__()
        
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim*2, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )
    
    def forward(self, user, movie):
        u = self.user_emb(user)
        m = self.movie_emb(movie)
        x = torch.cat([u, m], dim=1)
        return self.mlp(x)

In [ ]:
ncf_model_strong = StrongNCF(n_users, n_movies)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(ncf_model_strong.parameters(), lr=0.001)

In [ ]:
for epoch in range(10):
    total_loss = 0
    
    for users, movies, labels in train_loader_ncf:
        
        preds = ncf_model_strong(users, movies).squeeze()
        loss = loss_fn(preds, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print("Epoch", epoch, "Loss:", total_loss)

In [ ]:
def ncf_hit_rate(model, train_ncf, test_ncf, n_movies, k=10):
    model.eval()
    
    # build train watched dict
    train_df_temp = pd.DataFrame(train_ncf, columns=["user","movie"])
    
    hits = 0
    total = 0
    
    with torch.no_grad():
        for user_id, true_movie in test_ncf:
            
            user_tensor = torch.tensor([user_id] * n_movies)
            movie_tensor = torch.arange(n_movies)
            
            scores = model(user_tensor, movie_tensor).squeeze()
            
            watched = set(train_df_temp[train_df_temp.user == user_id].movie.values)
            
            candidates = []
            for i, s in enumerate(scores):
                if i not in watched:
                    candidates.append((i, s.item()))
            
            candidates.sort(key=lambda x: x[1], reverse=True)
            top_k = [x[0] for x in candidates[:k]]
            
            if true_movie in top_k:
                hits += 1
            
            total += 1
    
    return hits / total

In [ ]:
ncf_hr_new = ncf_hit_rate(ncf_model_strong, train_ncf, test_ncf, n_movies, k=10)
print("Strong NCF Hit@10:", ncf_hr_new)

In [29]:
seq_model = LSTMRecommender(n_movies, emb_dim=64, hidden_dim=128)
seq_model.load_state_dict(torch.load("lstm_model.pth"))
seq_model.eval()

LSTMRecommender(
  (movie_emb): Embedding(3533, 64)
  (lstm): LSTM(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=3533, bias=True)
)

In [30]:
def get_next_watch(model, sequence, k=10):
    seq_tensor = torch.tensor(sequence).unsqueeze(0)
    
    with torch.no_grad():
        scores = model(seq_tensor).squeeze()
    
    top_k = torch.topk(scores, k).indices.tolist()
    
    return top_k

In [42]:
def get_top_recommendations(model, user_history, k=10):
    sequence = user_history[-3:]
    
    seq_tensor = torch.tensor(sequence).unsqueeze(0)
    
    with torch.no_grad():
        scores = model(seq_tensor).squeeze()
    
    watched = set(user_history)
    
    candidates = []
    for i, s in enumerate(scores):
        if i not in watched:
            candidates.append((i, s.item()))
    
    candidates.sort(key=lambda x: x[1], reverse=True)
    
    filtered = candidates[5:5+k]
    
    return [x[0] for x in filtered]

In [33]:
import random

def cold_start(n_movies, k=10):
    return random.sample(range(n_movies), k)

In [34]:
user_history = [10, 25, 40]

print("Next Watch:", get_next_watch(seq_model, user_history))
print("Top Recommendations:", get_top_recommendations(seq_model, user_history))

Next Watch: [110, 73, 69, 32, 271, 31, 88, 163, 36, 324]
Top Recommendations: [110, 73, 69, 32, 271, 31, 88, 163, 36, 324]


In [36]:
def get_movie_titles(indices):
    movie_ids = [idx_to_movie[i] for i in indices]
    
    id_to_title = dict(zip(movies["movie"], movies["title"]))
    
    return [id_to_title[mid] for mid in movie_ids]

In [38]:
import pandas as pd

movies = pd.read_csv(
    "../data/ml-1m/movies.dat",
    sep="::",
    engine="python",
    names=["movie","title","genre"],
    encoding="latin-1"
)

print(type(movies))

<class 'pandas.core.frame.DataFrame'>


In [43]:
user_history = [10, 25, 40]

next_movies = get_next_watch(seq_model, user_history)
top_movies = get_top_recommendations(seq_model, user_history)

print("Next Watch:")
print(get_movie_titles(next_movies))

print("\nTop Recommendations:")
print(get_movie_titles(top_movies))

Next Watch:
['Forrest Gump (1994)', 'American Beauty (1999)', 'Terminator 2: Judgment Day (1991)', 'Toy Story (1995)', 'Groundhog Day (1993)', 'Apollo 13 (1995)', 'Matrix, The (1999)', 'Pulp Fiction (1994)', 'Star Wars: Episode IV - A New Hope (1977)', "There's Something About Mary (1998)"]

Top Recommendations:
['Apollo 13 (1995)', 'Matrix, The (1999)', 'Pulp Fiction (1994)', 'Star Wars: Episode IV - A New Hope (1977)', "There's Something About Mary (1998)", 'Braveheart (1995)', 'Full Monty, The (1997)', 'Princess Bride, The (1987)', 'Shawshank Redemption, The (1994)', 'Good Will Hunting (1997)']


In [7]:
import pickle

with open("movie_map.pkl", "wb") as f:
    pickle.dump(movie_map, f)